# Milestone 2 — Speech-to-Text
### Stage 4 of NOVA: converting the captured utterance into text with `faster-whisper`

This notebook teaches the concepts behind Milestone 2 before you rely on the `Transcriber` class you already have in `nova/audio/transcriber.py`. We'll cover:

1. What a speech-to-text model actually does (audio → tokens → text)
2. Why "faster-whisper" instead of plain OpenAI Whisper
3. Model size tradeoffs (tiny vs small vs medium vs large)
4. Feeding it real audio, and reading its output (segments, language, confidence)
5. The edge cases your `Transcriber` wrapper already handles for you

> **Note on this environment:** this sandbox has no microphone and no internet access to download model weights. Every cell either runs fully self-contained (using tiny synthetic arrays to show *shapes and interfaces*) or is clearly marked "run this on your own machine" for anything that needs a real downloaded model. Copy those cells into a local Jupyter session on your Windows machine, where `faster-whisper` will actually transcribe your voice.


In [ ]:
# Install on your own machine (not this sandbox):
# pip install faster-whisper numpy soundfile

import numpy as np
print("Ready.")


## 1. What is a speech-to-text model actually doing?

At a high level, Whisper (and models like it) work in three stages:

1. **Feature extraction** — the raw waveform (a 1D array of amplitudes) is converted into a **log-mel spectrogram**: a 2D image-like representation of *which frequencies are present at which moments in time*. This is the same idea as looking at a song's waveform vs. its spectrogram in an audio editor.
2. **Encoder** — a transformer encodes that spectrogram into a sequence of hidden representations, similar to how a text transformer encodes tokens.
3. **Decoder** — another transformer generates text tokens one at a time, conditioned on the encoded audio, until it produces an end-of-transcript token.

This is exactly why Whisper needs a fixed **input sample rate of 16kHz** — the model was trained on spectrograms computed at that rate. Feed it 44.1kHz audio without resampling and the timing/frequency information will be wrong.


In [ ]:
SAMPLE_RATE = 16000

# A log-mel spectrogram is expensive to compute correctly by hand,
# but here's the shape intuition using a crude substitute (just to visualize
# "waveform in, time x frequency grid out" -- NOT what Whisper's real mel filterbank does):

duration_s = 2.0
t = np.linspace(0, duration_s, int(SAMPLE_RATE * duration_s), endpoint=False)
fake_speech = 0.2 * np.sin(2 * np.pi * 150 * t) + 0.1 * np.random.randn(len(t))

print(f"Raw waveform shape: {fake_speech.shape}  (just a 1D array of amplitudes)")
print("Whisper internally turns this into an 80-channel x ~time-steps log-mel spectrogram")
print("before the encoder ever sees it -- faster-whisper does this step for you.")


## 2. Why `faster-whisper`, not plain `openai-whisper`?

`faster-whisper` re-implements Whisper's inference using **CTranslate2**, a fast inference engine (the same family of tricks used to speed up translation models). Practically, this gets you:

- **4-5x faster inference on CPU**, which matters a lot for a voice assistant that must respond quickly with no GPU
- **Lower memory usage** via quantization (`compute_type="int8"` in your config — trades a small amount of accuracy for a big speed/memory win)
- **The same model weights and accuracy** as the original Whisper — it's an inference engine swap, not a different model

This is exactly why `config.yaml` sets `stt.compute_type: "int8"` and `stt.device: "cpu"` — those two settings are the difference between NOVA responding in ~1s vs ~4s on a laptop with no GPU.


## 3. Model size tradeoffs

Whisper ships in several sizes. Bigger = more accurate, especially on accents/background noise/technical vocabulary, but slower and more memory-hungry. For an always-on assistant running on a laptop CPU, this tradeoff matters a lot:

| Size | Params | Relative speed (CPU) | When to use |
|---|---|---|---|
| tiny | 39M | fastest | quick prototyping, very simple commands |
| base | 74M | fast | short, simple commands |
| **small** | 244M | moderate | **NOVA's default — good balance for command-style speech** |
| medium | 769M | slow | dictation, longer sentences, noisy environments |
| large-v3 | 1.5B | slowest | offline batch transcription, not real-time assistants |

Your `config.yaml` currently sets `stt.model_size: "small"`. That's a deliberate choice for a voice *assistant* (short commands, needs to feel responsive) rather than a dictation tool (long-form, latency matters less).


In [ ]:
# --- RUN THIS ON YOUR OWN MACHINE ---
from faster_whisper import WhisperModel
import time

for size in ["tiny", "small"]:  # try more sizes yourself once you have the models downloaded
    model = WhisperModel(size, device="cpu", compute_type="int8")
    # Replace with a real short audio clip on your machine:
    # segments, info = model.transcribe("my_test_clip.wav")
    print(f"Loaded '{size}' model successfully")


## 4. Feeding it audio and reading the output

`faster-whisper`'s `.transcribe()` accepts either a **file path** or a **numpy float32 array** (which is exactly what your `SpeechSegmenter` hands back on `SPEECH_ENDED` — no file round-trip needed). It returns:

- `segments`: a **generator** of segment objects, each with `.text`, `.start`, `.end` (timestamps in seconds), and `.avg_logprob` (roughly, confidence)
- `info`: metadata about the whole transcription, including detected `.language` and `.language_probability`

Because `segments` is a generator, it's lazy — nothing is actually computed until you iterate over it. This is why your `Transcriber.transcribe()` method does `[seg.text.strip() for seg in segments]` — that list comprehension is what actually triggers the decoding.


In [ ]:
# --- RUN THIS ON YOUR OWN MACHINE, with a real .wav file ---
from faster_whisper import WhisperModel

model = WhisperModel("small", device="cpu", compute_type="int8")
segments, info = model.transcribe("my_test_clip.wav", language=None, vad_filter=False)

print(f"Detected language: {info.language} (confidence: {info.language_probability:.2f})")

for seg in segments:
    print(f"[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text}  (avg_logprob={seg.avg_logprob:.2f})")


### Why your `Transcriber` passes `vad_filter=False`

`faster-whisper` actually ships its *own* built-in VAD (to skip silent portions before transcribing). NOVA already ran Silero VAD in Milestone 1 to decide when the utterance started and ended — running a second VAD pass here would just discard the same silence twice for no benefit, and can occasionally clip the start/end of short commands. So `Transcriber` explicitly disables it and trusts the audio boundaries `SpeechSegmenter` already computed.


In [ ]:
# Simulate what your Transcriber class does internally, with fake segments
# (this cell runs fully in this sandbox -- no model needed)

class FakeSegment:
    def __init__(self, text, start, end, avg_logprob):
        self.text, self.start, self.end, self.avg_logprob = text, start, end, avg_logprob

class FakeInfo:
    def __init__(self, language="en", language_probability=0.98):
        self.language = language
        self.language_probability = language_probability

fake_segments = [
    FakeSegment(" hey nova ", 0.0, 0.6, -0.12),
    FakeSegment(" what's the weather today ", 0.6, 2.1, -0.20),
]
fake_info = FakeInfo()

# This is exactly the logic inside Transcriber.transcribe():
text_parts = [seg.text.strip() for seg in fake_segments]
transcript = " ".join(part for part in text_parts if part)

print("Reconstructed transcript:", repr(transcript))
print(f"Detected language: {fake_info.language} ({fake_info.language_probability:.0%} confidence)")


## 5. Edge cases your `Transcriber` wrapper already handles

Look back at `nova/audio/transcriber.py` — two defensive choices matter in practice:

1. **Empty audio guard**: `if audio.size == 0: return ""`. Your `SpeechSegmenter` should never emit a zero-length utterance, but defending against it means a bug upstream fails safely (empty string) instead of crashing the whole assistant on a malformed frame.
2. **Filtering blank segments**: `if part` inside the join. Whisper occasionally emits a segment that's just whitespace or a stray token during background noise — silently dropping empty segments avoids NOVA responding to nothing with an oddly-spaced reply.

Both of these are exactly what `tests/test_milestone2_stt.py` verifies with `FakeWhisperModel` — no real model needed to prove the wrapper logic is correct.


In [ ]:
# Try it yourself: what happens with a segment that's ONLY whitespace?
text_parts = [seg.text.strip() for seg in [FakeSegment("hello", 0, 1, -0.1), FakeSegment("   ", 1, 1.1, -0.9), FakeSegment("there", 1.1, 1.5, -0.15)]]
transcript = " ".join(part for part in text_parts if part)
print(repr(transcript))  # should be 'hello there', not 'hello  there' with a double space


## 6. Putting Milestone 1 + 2 together

```
SpeechSegmenter emits SPEECH_ENDED with utterance audio (float32, 16kHz)
        │
        ▼
Transcriber.transcribe(audio)
        │
        ├─ audio.size == 0?  → return ""  (defensive guard)
        │
        ▼
WhisperModel.transcribe(audio, vad_filter=False)   # no double-VAD
        │
        ▼
[seg.text.strip() for seg in segments]   # triggers lazy decoding
        │
        ▼
" ".join(non-empty parts)  → clean transcript string
        │
        ▼
handed to Milestone 3's LLM brain as the user's message
```

### Exercises
1. Record 3 short `.wav` clips on your machine at different volumes (whisper, normal, loud) and compare `avg_logprob` across them — does confidence drop for the quiet one?
2. Try `model_size="tiny"` vs `"small"` on the same clip and time both with `time.perf_counter()` — how much slower is `small`, and is the transcript actually more accurate for your voice?
3. Deliberately mumble a command with background music playing, transcribe it, and look at `info.language_probability` — low confidence there is a good signal NOVA could use later to ask "sorry, could you repeat that?" instead of guessing.

**Next notebook:** Milestone 3 — the NOVA Brain (local LLM via Ollama + agent/tool-routing loop). Say the word and I'll build it.
